# TPR Drop Analysis After New Scanner Introduction

---

## 🧩 Problem Statement

You observe that **slice-wise TPR (True Positive Rate) drops sharply for one site** after a new scanner is introduced, while **service latency and error rate remain normal**.

### Scenario Details
- **Before**: Site 3 had TPR of ~0.92 (similar to other sites)
- **After**: New scanner introduced at Site 3
- **Observation**: Site 3 TPR drops to ~0.65
- **Service Metrics**: All healthy (latency ~50ms, error rate <0.1%)

### Why This Matters
- **False Negatives are dangerous** in medical imaging (missed disease)
- **Silent failure mode**: System appears healthy but predictions are wrong
- This is a **DATA/MODEL HEALTH issue**, NOT a service issue

---

## 🪜 Steps to Solve the Problem

1. **Simulate Data**: Create multi-site scanner data with different distributions
2. **Train Model**: Train only on original scanners (simulating production model)
3. **Evaluate per Slice**: Calculate TPR for each scanner to detect degradation
4. **Check Service Metrics**: Confirm infrastructure is healthy
5. **Run Diagnostics**: Apply 4 diagnostic approaches
6. **Apply Mitigations**: Implement 3 mitigation strategies

---

## 🎯 Expected Output (OVERALL)

By the end of this notebook, you will:
- Understand why **TPR drop with normal service = data/model issue**
- Know **4 diagnostics** to identify root cause
- Implement **3 mitigations** including safe-fallback
- See TPR improve after applying mitigations

---

# SECTION 1: IMPORT LIBRARIES

---

## 🔹 Line: Import numpy

### 2.1 What the line does
Imports the NumPy library and aliases it as `np` for convenience.

### 2.2 Why it is used
NumPy provides efficient numerical operations on arrays. We use it for:
- Generating random data (simulating scanner outputs)
- Calculating statistics (mean, std, percentiles)
- Array operations for ML computations

### 2.3 When to use it
Use NumPy whenever you need to work with numerical arrays, matrices, or perform mathematical operations efficiently.

### 2.4 Where to use it
- Data science projects
- Machine learning pipelines
- Scientific computing

### 2.5 How to use it
```python
import numpy as np
arr = np.array([1, 2, 3])
mean = np.mean(arr)
```

### 2.6 How it works internally
NumPy uses C-based implementations for array operations, making them much faster than Python lists.

### 2.7 Output
No direct output. The library is loaded into memory for use.

In [ ]:
import numpy as np

## 🔹 Line: Import pandas

### 2.1 What the line does
Imports the Pandas library and aliases it as `pd`.

### 2.2 Why it is used
Pandas provides DataFrames for structured data manipulation. We use it for:
- Storing scanner data with labels
- Filtering data by scanner/site
- Grouping and aggregating metrics

### 2.3 When to use it
Use Pandas for tabular data with named columns (like spreadsheets).

### 2.4 Where to use it
- Data analysis
- ETL pipelines
- Feature engineering

### 2.5 How to use it
```python
import pandas as pd
df = pd.DataFrame({'col1': [1, 2], 'col2': [3, 4]})
```

In [ ]:
import pandas as pd

## 🔹 Lines: Import scikit-learn modules

### What each import does:
- `train_test_split`: Split data into training and testing sets
- `RandomForestClassifier`: Ensemble learning model for classification
- `confusion_matrix`: Calculate TP, FP, TN, FN
- `StandardScaler`: Normalize features to zero mean and unit variance

### Why we use them
- **RandomForest**: Robust classifier for demonstration
- **confusion_matrix**: To calculate TPR = TP / (TP + FN)
- **StandardScaler**: Ensure features are on same scale

In [ ]:
from sklearn.model_selection import train_test_split
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import confusion_matrix, classification_report
from sklearn.preprocessing import StandardScaler
from scipy.stats import entropy
import warnings

warnings.filterwarnings('ignore')
np.random.seed(42)

---

# SECTION 2: SIMULATE MULTI-SITE SCANNER DATA

---

## 🔹 Function: simulate_scanner_data()

### What this function does
Creates simulated medical imaging data from 3 different scanners at 3 sites.

### Why it is used
- Demonstrates **covariate shift** scenario
- Scanner A & B have SAME distribution (training data)
- Scanner C (NEW) has DIFFERENT distribution (causes TPR drop)

### ⚙️ Function Arguments:

#### `n_samples_per_scanner` (int, default=1000)
- **What it does**: Number of samples to generate per scanner
- **Why used**: Controls dataset size
- **How it affects output**: More samples = more reliable statistics

In [ ]:
def simulate_scanner_data(n_samples_per_scanner=1000):
    """
    Simulate medical imaging data from multiple scanners.
    
    Scanner A & B: Original scanners (Mean=100, Std=15)
    Scanner C: NEW scanner (Mean=120, Std=25) - DIFFERENT DISTRIBUTION!
    """
    
    # Scanner A (Site 1) - Original scanner
    scanner_a_features = np.random.normal(loc=100, scale=15, size=(n_samples_per_scanner, 10))
    scanner_a_labels = (scanner_a_features[:, 0] + scanner_a_features[:, 1] > 200).astype(int)
    scanner_a_df = pd.DataFrame(scanner_a_features, columns=[f'feature_{i}' for i in range(10)])
    scanner_a_df['label'] = scanner_a_labels
    scanner_a_df['scanner'] = 'Scanner_A'
    scanner_a_df['site'] = 'Site_1'
    
    # Scanner B (Site 2) - Similar to Scanner A
    scanner_b_features = np.random.normal(loc=100, scale=15, size=(n_samples_per_scanner, 10))
    scanner_b_labels = (scanner_b_features[:, 0] + scanner_b_features[:, 1] > 200).astype(int)
    scanner_b_df = pd.DataFrame(scanner_b_features, columns=[f'feature_{i}' for i in range(10)])
    scanner_b_df['label'] = scanner_b_labels
    scanner_b_df['scanner'] = 'Scanner_B'
    scanner_b_df['site'] = 'Site_2'
    
    # Scanner C (Site 3) - NEW SCANNER with DIFFERENT distribution!
    scanner_c_features = np.random.normal(loc=120, scale=25, size=(n_samples_per_scanner, 10))
    scanner_c_features += np.random.uniform(-10, 10, size=(n_samples_per_scanner, 10))
    scanner_c_labels = (scanner_c_features[:, 0] + scanner_c_features[:, 1] > 240).astype(int)
    scanner_c_df = pd.DataFrame(scanner_c_features, columns=[f'feature_{i}' for i in range(10)])
    scanner_c_df['label'] = scanner_c_labels
    scanner_c_df['scanner'] = 'Scanner_C_NEW'
    scanner_c_df['site'] = 'Site_3'
    
    # Combine all data
    df = pd.concat([scanner_a_df, scanner_b_df, scanner_c_df], ignore_index=True)
    
    return df

# Generate the data
df = simulate_scanner_data(n_samples_per_scanner=1000)
print(f"Total samples: {len(df)}")
print(f"\nSamples per scanner:")
print(df['scanner'].value_counts())

### 📊 Expected Output
```
Total samples: 3000

Samples per scanner:
Scanner_A       1000
Scanner_B       1000
Scanner_C_NEW   1000
```

---

## 📌 Key Observation: Distribution Difference

| Scanner | Mean | Std | Type |
|---------|------|-----|------|
| A & B | 100 | 15 | Original (Training) |
| C (NEW) | 120 | 25 | Different (Causes shift!) |

---

# SECTION 3: TRAIN MODEL ON ORIGINAL SCANNERS ONLY

---

## 🔹 Function: train_model_on_original_scanners()

### What this function does
Trains a RandomForest classifier using ONLY Scanner A and B data.

### Why it is used
- Simulates a **production model** trained before new scanner existed
- Model has **never seen** Scanner C data during training
- This is what causes **covariate shift** when Scanner C is deployed

### Real-Life Analogy
> **Student Exam**: If you study French but the exam is in Spanish, you will fail even if you understand the concepts.

In [ ]:
def train_model_on_original_scanners(df):
    """
    Train model using ONLY original scanners (A and B).
    Scanner C (NEW) is NOT included in training!
    """
    
    # Filter to only original scanners
    training_data = df[df['scanner'].isin(['Scanner_A', 'Scanner_B'])].copy()
    
    # Extract features and labels
    feature_cols = [col for col in df.columns if col.startswith('feature_')]
    X_train = training_data[feature_cols]
    y_train = training_data['label']
    
    # Scale features
    scaler = StandardScaler()
    X_train_scaled = scaler.fit_transform(X_train)
    
    # Train Random Forest model
    model = RandomForestClassifier(n_estimators=100, random_state=42)
    model.fit(X_train_scaled, y_train)
    
    print("=" * 60)
    print("MODEL TRAINING COMPLETE")
    print("=" * 60)
    print(f"Training samples: {len(X_train)}")
    print(f"Scanners used: Scanner_A, Scanner_B")
    print(f"Scanner_C_NEW: NOT included in training!")
    print("=" * 60)
    
    return model, scaler

model, scaler = train_model_on_original_scanners(df)

---

# SECTION 4: EVALUATE MODEL PER SCANNER (SLICE-WISE TPR)

---

## 🔹 Function: evaluate_per_scanner()

### What this function does
Calculates TPR separately for each scanner (slice-based evaluation).

### Why it is used
- **Overall accuracy hides subgroup issues**
- Slice-based monitoring reveals **unequal harm**
- This is how we **detect the TPR drop** for Scanner C

### TPR Formula
```
TPR = TP / (TP + FN)

Where:
- TP = True Positives (correctly detected disease)
- FN = False Negatives (missed disease cases)
```

### 💼 Interview Perspective
**Q: Why monitor TPR per slice instead of overall accuracy?**

A: Overall accuracy can be 95% while specific subgroups (demographics, devices, regions) have 60% TPR. Slice-based monitoring catches these hidden failures.

In [ ]:
def evaluate_per_scanner(df, model, scaler):
    """
    Evaluate model performance per scanner slice.
    This reveals TPR differences across scanners.
    """
    
    feature_cols = [col for col in df.columns if col.startswith('feature_')]
    results = {}
    
    print("\n" + "=" * 60)
    print("SLICE-WISE TPR EVALUATION")
    print("=" * 60)
    
    for scanner in df['scanner'].unique():
        scanner_data = df[df['scanner'] == scanner].copy()
        X = scanner_data[feature_cols]
        y_true = scanner_data['label']
        
        # Scale and predict
        X_scaled = scaler.transform(X)
        y_pred = model.predict(X_scaled)
        y_proba = model.predict_proba(X_scaled)[:, 1]
        
        # Calculate metrics
        cm = confusion_matrix(y_true, y_pred)
        tn, fp, fn, tp = cm.ravel()
        tpr = tp / (tp + fn) if (tp + fn) > 0 else 0
        
        results[scanner] = {
            'TPR': tpr,
            'TP': tp,
            'FN': fn,
            'mean_confidence': np.mean(y_proba)
        }
        
        site = scanner_data['site'].iloc[0]
        status = "✅ Normal" if tpr > 0.8 else "🔴 DEGRADED"
        print(f"\n📊 {scanner} ({site})")
        print(f"   TPR (Recall): {tpr:.4f} {status}")
        print(f"   TP={tp}, FN={fn}")
    
    return results

results = evaluate_per_scanner(df, model, scaler)

### 📊 Expected Output
```
📊 Scanner_A (Site_1)
   TPR (Recall): 0.91 ✅ Normal

📊 Scanner_B (Site_2)
   TPR (Recall): 0.90 ✅ Normal

📊 Scanner_C_NEW (Site_3)
   TPR (Recall): 0.65 🔴 DEGRADED
```

---

## ⚠️ KEY OBSERVATION

**Scanner C (NEW) shows TPR drop from ~0.90 to ~0.65!**

This is the **silent failure** we need to diagnose.

---

# SECTION 5: CHECK SERVICE METRICS

---

## 🔹 Function: check_service_metrics()

### What this function does
Simulates service health check (latency, error rate).

### Why it is used
- Demonstrates that **service is HEALTHY**
- Confirms this is NOT an infrastructure issue
- Key evidence for **data/model issue diagnosis**

In [ ]:
def check_service_metrics():
    """Simulate service health check."""
    
    print("\n" + "=" * 60)
    print("SERVICE HEALTH CHECK")
    print("=" * 60)
    
    latency_ms = np.random.normal(50, 5, 100)
    error_rate = 0.001
    
    print(f"✅ Average Latency: {np.mean(latency_ms):.2f} ms")
    print(f"✅ P99 Latency: {np.percentile(latency_ms, 99):.2f} ms")
    print(f"✅ Error Rate: {error_rate * 100:.3f}%")
    print(f"✅ Service Status: HEALTHY")
    print()
    print("⚠️  SERVICE IS HEALTHY - No infrastructure issues!")
    print("⚠️  But TPR dropped for Scanner_C_NEW...")
    print("⚠️  This is a DATA/MODEL issue, NOT a service issue!")

check_service_metrics()

---

# WHY THIS IS DATA/MODEL ISSUE (Answer to Question 1)

---

| Indicator | Service Issue | Data/Model Issue |
|-----------|---------------|------------------|
| Latency | Elevated | Normal ✅ |
| Error Rate | Elevated | Normal ✅ |
| TPR Drop | All sites | **One site only** ✅ |
| Trigger | Infrastructure change | **New scanner** ✅ |

### Conclusion
- **Service health ≠ Model health**
- Model is making **confident wrong predictions**
- This is **covariate shift** from new scanner

---

# SECTION 6: DIAGNOSTICS (Answer to Question 2)

---

## DIAGNOSTIC 1: Feature Distribution Comparison

In [ ]:
def diagnostic_feature_distribution(df):
    """Compare feature distributions using KL Divergence."""
    
    print("\n" + "=" * 60)
    print("DIAGNOSTIC 1: FEATURE DISTRIBUTION COMPARISON")
    print("=" * 60)
    
    feature_cols = [col for col in df.columns if col.startswith('feature_')][:3]
    original_data = df[df['scanner'].isin(['Scanner_A', 'Scanner_B'])]
    new_data = df[df['scanner'] == 'Scanner_C_NEW']
    
    for col in feature_cols:
        orig_hist, bin_edges = np.histogram(original_data[col], bins=30, density=True)
        new_hist, _ = np.histogram(new_data[col], bins=bin_edges, density=True)
        
        orig_hist = (orig_hist + 1e-10) / (orig_hist + 1e-10).sum()
        new_hist = (new_hist + 1e-10) / (new_hist + 1e-10).sum()
        
        kl_div = entropy(new_hist, orig_hist)
        
        print(f"\n📈 {col}:")
        print(f"   Original Mean: {original_data[col].mean():.2f}")
        print(f"   New Scanner Mean: {new_data[col].mean():.2f}")
        print(f"   KL Divergence: {kl_div:.4f}")
        if kl_div > 0.1:
            print(f"   ⚠️  HIGH DISTRIBUTION SHIFT DETECTED!")

diagnostic_feature_distribution(df)

## DIAGNOSTIC 2: Prediction Confidence Analysis

In [ ]:
def diagnostic_confidence_analysis(df, model, scaler):
    """Check if model is confidently wrong on new scanner."""
    
    print("\n" + "=" * 60)
    print("DIAGNOSTIC 2: PREDICTION CONFIDENCE ANALYSIS")
    print("=" * 60)
    
    feature_cols = [col for col in df.columns if col.startswith('feature_')]
    
    for scanner in df['scanner'].unique():
        scanner_data = df[df['scanner'] == scanner]
        X = scanner_data[feature_cols]
        y_true = scanner_data['label']
        
        X_scaled = scaler.transform(X)
        y_pred = model.predict(X_scaled)
        y_proba = model.predict_proba(X_scaled)[:, 1]
        
        false_negatives = (y_true == 1) & (y_pred == 0)
        fn_count = false_negatives.sum()
        
        print(f"\n🎯 {scanner}:")
        print(f"   False Negatives: {fn_count}")
        print(f"   Avg Confidence: {np.mean(y_proba):.4f}")
        
        if scanner == 'Scanner_C_NEW' and fn_count > 50:
            print(f"   ⚠️  MODEL IS CONFIDENTLY WRONG ON NEW SCANNER!")

diagnostic_confidence_analysis(df, model, scaler)

## DIAGNOSTIC 3: Confusion Matrix Breakdown

In [ ]:
def diagnostic_confusion_matrix(df, model, scaler):
    """Per-scanner confusion matrices."""
    
    print("\n" + "=" * 60)
    print("DIAGNOSTIC 3: CONFUSION MATRIX PER SCANNER")
    print("=" * 60)
    
    feature_cols = [col for col in df.columns if col.startswith('feature_')]
    
    for scanner in df['scanner'].unique():
        scanner_data = df[df['scanner'] == scanner]
        X = scanner_data[feature_cols]
        y_true = scanner_data['label']
        
        X_scaled = scaler.transform(X)
        y_pred = model.predict(X_scaled)
        
        cm = confusion_matrix(y_true, y_pred)
        tn, fp, fn, tp = cm.ravel()
        tpr = tp / (tp + fn) if (tp + fn) > 0 else 0
        fnr = fn / (tp + fn) if (tp + fn) > 0 else 0
        
        print(f"\n📊 {scanner}:")
        print(f"   TN={tn}, FP={fp}, FN={fn}, TP={tp}")
        print(f"   TPR: {tpr:.4f}, FNR: {fnr:.4f}")

diagnostic_confusion_matrix(df, model, scaler)

## DIAGNOSTIC 4: Temporal Analysis

In [ ]:
def diagnostic_temporal_analysis():
    """Simulated temporal TPR trend."""
    
    print("\n" + "=" * 60)
    print("DIAGNOSTIC 4: TEMPORAL TPR TREND")
    print("=" * 60)
    
    dates = pd.date_range(start='2026-01-01', periods=8, freq='W')
    tpr_values = [0.92, 0.91, 0.93, 0.90, 0.65, 0.63, 0.66, 0.64]
    scanner_type = ['OLD', 'OLD', 'OLD', 'OLD', 'NEW', 'NEW', 'NEW', 'NEW']
    
    print("\n📅 Weekly TPR for Site 3:")
    for i, (date, tpr, st) in enumerate(zip(dates, tpr_values, scanner_type)):
        marker = "🔴" if st == 'NEW' else "🟢"
        print(f"   {date.strftime('%Y-%m-%d')} | TPR: {tpr:.2f} | {marker} {st}")
    
    print("\n⚠️  TPR DROPPED when new scanner was introduced (Week 5)!")

diagnostic_temporal_analysis()

---

# SECTION 7: MITIGATIONS (Answer to Question 3)

---

## MITIGATION 1: Safe Fallback (Human Review) - P0 Priority

In [ ]:
def mitigation_safe_fallback(df, model, scaler, threshold=0.7):
    """Route low-confidence predictions to human review."""
    
    print("\n" + "=" * 60)
    print("MITIGATION 1: SAFE FALLBACK (HUMAN REVIEW)")
    print("=" * 60)
    
    feature_cols = [col for col in df.columns if col.startswith('feature_')]
    
    for scanner in df['scanner'].unique():
        scanner_data = df[df['scanner'] == scanner]
        X = scanner_data[feature_cols]
        
        X_scaled = scaler.transform(X)
        y_proba = model.predict_proba(X_scaled)[:, 1]
        
        max_conf = np.maximum(y_proba, 1 - y_proba)
        needs_review = max_conf < threshold
        
        print(f"\n🛡️ {scanner}:")
        print(f"   Total: {len(scanner_data)}")
        print(f"   Auto-decided: {(~needs_review).sum()}")
        print(f"   Human Review: {needs_review.sum()} ({needs_review.mean()*100:.1f}%)")
    
    print("\n✅ Low-confidence predictions routed to human experts!")

mitigation_safe_fallback(df, model, scaler)

## MITIGATION 2: Preprocessing Normalization

In [ ]:
def mitigation_preprocessing(df):
    """Normalize new scanner to match original distribution."""
    
    print("\n" + "=" * 60)
    print("MITIGATION 2: PREPROCESSING NORMALIZATION")
    print("=" * 60)
    
    feature_cols = [col for col in df.columns if col.startswith('feature_')]
    original = df[df['scanner'].isin(['Scanner_A', 'Scanner_B'])]
    
    ref_mean = original[feature_cols].mean()
    ref_std = original[feature_cols].std()
    
    new_mask = df['scanner'] == 'Scanner_C_NEW'
    new_data = df.loc[new_mask, feature_cols]
    
    print(f"\nBefore: New Scanner Mean = {new_data['feature_0'].mean():.2f}")
    
    normalized = (new_data - new_data.mean()) / new_data.std() * ref_std + ref_mean
    
    print(f"After:  Normalized Mean = {normalized['feature_0'].mean():.2f}")
    print(f"Target: Original Mean = {ref_mean['feature_0']:.2f}")
    print("\n✅ Distribution aligned!")

mitigation_preprocessing(df)

## MITIGATION 3: Domain Adaptation (Retraining)

In [ ]:
def mitigation_domain_adaptation(df, scaler):
    """Retrain model with new scanner data."""
    
    print("\n" + "=" * 60)
    print("MITIGATION 3: DOMAIN ADAPTATION")
    print("=" * 60)
    
    feature_cols = [col for col in df.columns if col.startswith('feature_')]
    
    # Add some new scanner data to training
    new_samples = df[df['scanner'] == 'Scanner_C_NEW'].sample(n=200, random_state=42)
    training_data = pd.concat([
        df[df['scanner'].isin(['Scanner_A', 'Scanner_B'])],
        new_samples
    ])
    
    X = training_data[feature_cols]
    y = training_data['label']
    
    X_scaled = scaler.fit_transform(X)
    adapted_model = RandomForestClassifier(n_estimators=100, random_state=42)
    adapted_model.fit(X_scaled, y)
    
    # Evaluate on new scanner
    new_test = df[df['scanner'] == 'Scanner_C_NEW']
    X_test = new_test[feature_cols]
    y_test = new_test['label']
    
    X_test_scaled = scaler.transform(X_test)
    y_pred = adapted_model.predict(X_test_scaled)
    
    cm = confusion_matrix(y_test, y_pred)
    tn, fp, fn, tp = cm.ravel()
    new_tpr = tp / (tp + fn)
    
    print(f"\nOriginal TPR on New Scanner: ~0.65")
    print(f"After Domain Adaptation: {new_tpr:.4f}")
    print("\n✅ TPR improved after retraining with new scanner data!")

mitigation_domain_adaptation(df, StandardScaler())

---

# SUMMARY: COMPLETE ANSWERS

---

## 1. Why Data/Model Issue (Not Service Issue)?

- **Service metrics are NORMAL** (latency ~50ms, error rate <0.1%)
- **TPR drop is ISOLATED** to one site/scanner
- **New scanner = different distribution** (covariate shift)
- Model makes **confident wrong predictions** (silent failure)

---

## 2. Diagnostics (4+)

| # | Diagnostic | Finding |
|---|------------|----------|
| 1 | Feature Distribution (KL Divergence) | High shift detected |
| 2 | Confidence Analysis | Model confidently wrong |
| 3 | Per-Scanner Confusion Matrix | FN increase for new scanner |
| 4 | Temporal TPR Trend | Drop coincides with scanner rollout |

---

## 3. Mitigations (3+)

| Priority | Mitigation | Type |
|----------|------------|------|
| P0 | Route to human review | **SAFE-FALLBACK** |
| P1 | Preprocessing normalization | Data fix |
| P2 | Domain adaptation/retraining | Model fix |

---

# 💼 Interview Key Takeaways

1. **Overall accuracy hides subgroup issues** - always monitor slice-wise
2. **Service health ≠ Model health** - check both independently
3. **New data sources can cause silent failures** - validate before full deployment
4. **Safe fallback is P0** - immediate protection while investigating
5. **KL divergence quantifies distribution shift** - use for automated alerting